# 07 · The plotting ecosystem

**Session 8 — Space and Time (homework)** · about 45 minutes

**You will learn to**
- choose between Matplotlib, Seaborn, Plotly and Bokeh for a given figure
- use Seaborn's figure-level functions and its objects interface
- check that a legend tells the truth
- make interactive charts and maps with Plotly and Bokeh

**Data:** fMRI signals (seaborn example data); electricity consumption by municipality (Transparència Catalunya); comarques (ICGC).

**Outline**
0. Before you start (no AI)
1. Gallery challenge
2. Matplotlib vs Seaborn
3. Interactive: Plotly
4. Interactive maps: Bokeh
5. Your data

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

For each figure, which library would you pick and why?
(a) a figure for a journal article; (b) exploring 20 variables quickly; (c) a map people can hover on a web page.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

## 1. Gallery challenge

Five canonical plots on synthetic data. **Homework:** redo them with your project data.

In [ ]:
rng = np.random.default_rng(0)
n = 200
df = pd.DataFrame({"x": np.linspace(0, 10, n)})
df["y"] = np.sin(df["x"]) + 0.2 * rng.standard_normal(n)
df["category"] = rng.choice(list("ABC"), size=n)

fig, axs = plt.subplots(2, 2, figsize=(9, 6), layout="constrained")
axs[0, 0].plot(df["x"], df["y"])
axs[0, 0].set(title="Line", xlabel="x", ylabel="y")
sns.histplot(df["y"], kde=True, ax=axs[0, 1]).set(title="Histogram + KDE")
sns.boxplot(data=df, x="category", y="y", ax=axs[1, 0]).set(title="Box plot by category")
sns.stripplot(data=df, x="category", y="y", ax=axs[1, 0], color="black", size=2)
sns.heatmap(df[["x", "y"]].corr(), annot=True, vmin=-1, vmax=1, cmap="RdBu_r", ax=axs[1, 1])
axs[1, 1].set_title("Correlation")
plt.show()

The box plot shows the points too: a box alone hides how many observations there are.

## 2. Matplotlib vs Seaborn

fMRI signal over time for two brain regions and two event types (seaborn's example dataset).

In [ ]:
fmri = pd.read_csv(DATA / "examples/fmri.csv")
fmri.head()

**Seaborn** aggregates, draws the confidence band and builds the legend in one call:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.lineplot(data=fmri, x="timepoint", y="signal", hue="region", style="event", ax=ax)
plt.show()

The same figure with **Matplotlib** needs every step by hand: grouping, mean, 95% interval, colours,
line styles — and the legend.

### Verify the AI

The code below was written by hand and polished with an AI assistant. It runs and looks fine.
**Compare its legend with the Seaborn figure above.** Is it telling the truth?

In [ ]:
from matplotlib.lines import Line2D
from scipy.stats import sem

colors = {"parietal": "C0", "frontal": "C1"}
styles = {"stim": "-", "cue": "--"}

fig, ax = plt.subplots(figsize=(6, 4))
for (region, event), g in fmri.groupby(["region", "event"]):
    stats = g.groupby("timepoint")["signal"].agg(["mean", sem])
    ax.plot(stats.index, stats["mean"], color=colors[region], ls=styles[event])
    ax.fill_between(stats.index, stats["mean"] - 1.96 * stats["sem"],
                    stats["mean"] + 1.96 * stats["sem"], color=colors[region], alpha=0.2)

region_handles = [Line2D([], [], color=c) for c in colors.values()]
region_legend = ax.legend(region_handles, ["frontal", "parietal"], title="region", loc="upper right")
ax.add_artist(region_legend)
ax.legend([Line2D([], [], color="black", ls=s) for s in styles.values()], list(styles),
          title="event", loc="center right")
ax.set(xlabel="timepoint", ylabel="signal")
plt.show()

### TASK I — make the legend tell the truth

Rebuild the figure so each legend label is generated **from the same dictionary** that sets the colours
(never typed separately). Keep the figure in `fig_fixed`.

In [ ]:
# TASK I — your code here
fig_fixed = None

In [ ]:
# ✅ Self-check TASK I
from matplotlib.colors import to_hex

legends = [a for a in fig_fixed.axes[0].get_children() if type(a).__name__ == "Legend"]
region_leg = next(lg for lg in legends if lg.get_title().get_text() == "region")
for handle, text in zip(region_leg.legend_handles, region_leg.get_texts()):
    assert to_hex(handle.get_color()) == to_hex(colors[text.get_text()]), f"{text.get_text()} has the wrong colour"
print("✅ TASK I passed: the legend matches the data")

Seaborn's **objects interface** builds a plot from layers, like a grammar of graphics:

In [ ]:
import seaborn.objects as so

(
    so.Plot(fmri, x="timepoint", y="signal", color="region", linestyle="event")
    .add(so.Line(), so.Agg())
    .add(so.Band(), so.Est(errorbar=("ci", 95)), group="event")
    .label(x="Time point", y="Signal")
)

## 3. Interactive: Plotly

`plotly.express` makes interactive figures (hover, zoom) that work in notebooks and HTML pages.

In [ ]:
import plotly.express as px

means = fmri.groupby(["timepoint", "region", "event"], as_index=False)["signal"].mean()
px.line(means, x="timepoint", y="signal", color="region", line_dash="event",
        title="fMRI signal (mean per time point)")

An interactive choropleth: electricity consumption per km² by comarca (2023).

In [ ]:
import geopandas as gpd

energy = pd.read_parquet(DATA / "examples/energy_consumption_municipalities.parquet").query("any == 2023")
municipis = gpd.read_file(DATA / "icgc/divisions-administratives-v2r1-municipis-1000000-20250730.json")
comarques = gpd.read_file(DATA / "icgc/divisions-administratives-v2r1-comarques-250000-20250730.json")

energy["ine5"] = energy["cdmun"].astype(str).str.zfill(5)
municipis["ine5"] = municipis["CODIMUNI"].str[:5]
per_comarca = (
    energy.merge(municipis[["ine5", "CODICOMAR"]], on="ine5", how="left", validate="many_to_one")
    .groupby("CODICOMAR", as_index=False)["consum_kwh"].sum()
)
comarques = comarques.merge(per_comarca, on="CODICOMAR", how="left", validate="one_to_one")
comarques["gwh_per_km2"] = comarques["consum_kwh"] / 1e6 / comarques["AREAC5000"]

fig = px.choropleth_map(
    comarques, geojson=comarques.__geo_interface__, locations="CODICOMAR",
    featureidkey="properties.CODICOMAR", color="gwh_per_km2", color_continuous_scale="Viridis",
    hover_name="NOMCOMAR", center={"lat": 41.7, "lon": 1.7}, zoom=6.3, map_style="carto-positron",
    labels={"gwh_per_km2": "GWh / km²"},
)
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig

## 4. Interactive maps: Bokeh

Bokeh builds interactive web graphics from Python. (The old `pandas-bokeh` shortcut no longer works with
current Bokeh versions, so we use Bokeh directly.)

In [ ]:
from bokeh.io import output_notebook, show
from bokeh.models import ColorBar, GeoJSONDataSource, HoverTool, LinearColorMapper
from bokeh.palettes import Viridis256
from bokeh.plotting import figure

output_notebook(hide_banner=True)

gdf = comarques.to_crs(25831)[["NOMCOMAR", "gwh_per_km2", "geometry"]]
source = GeoJSONDataSource(geojson=gdf.to_json())
mapper = LinearColorMapper(palette=Viridis256, low=gdf["gwh_per_km2"].min(), high=gdf["gwh_per_km2"].max())

p = figure(title="Electricity consumption per km², 2023", match_aspect=True, width=600, height=550,
           toolbar_location="right")
p.patches("xs", "ys", source=source, fill_color={"field": "gwh_per_km2", "transform": mapper},
          line_color="white", line_width=0.5)
p.add_tools(HoverTool(tooltips=[("Comarca", "@NOMCOMAR"), ("GWh/km²", "@gwh_per_km2{0.00}")]))
p.add_layout(ColorBar(color_mapper=mapper, title="GWh/km²"), "right")
p.axis.visible = False
p.grid.visible = False
show(p)

Barcelona dominates the colour scale: one very dense comarca flattens all the others. A log or
quantile scale (notebook 08) would show more. **Ask yourself the same question about your maps.**

## 5. Your data (homework)

Redo the gallery of section 1 with your project data, then pick **one** figure for your article and
write, in one sentence, why you chose that library for it.

**Next:** `08_geospatial_maps.ipynb`.